#  `AIHUB 186.복지_분야_콜센터_상담데이터` 텍스트데이터 정보 획득

* 입력 폴더 위치: /home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터
* 출력 폴더 위치: /home/data/expr/week2/data_prepare_aihub
* 출력 파일: orgtext.tsv
  * 탭으로 구분된 csv 파일 (tsv)
  * 열
      * file_label
      * category1
      * category2
      * category3
      * speaker_type
      * orgtext

### 입력폴더, 출력파일, 에러파일 정의

In [20]:
input_dir = "/home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터"
output_file = "/home/data/expr/week2/03-data_prepare_aihub/orgtext.tsv"
error_file = "/home/data/expr/week2/03-data_prepare_aihub/orgtext_errors.tsv"

### 라이브러리 로드

In [21]:
from pathlib import Path
import json
import csv
from collections import Counter
from typing import Any

### 함수 정의

In [22]:

def extract_orgtext_from_folder(
    input_dir: str,
    output_file: str = "orgtext.tsv",
    error_file: str = "orgtext_errors.tsv",
    include_relative_path: bool = False,
) -> None:
    """
    input_dir 이하의 모든 JSON 파일을 재귀적으로 탐색하여
    오류가 없는 JSON 파일만 TSV로 저장합니다.

    저장 열:
    - file_label
    - category1
    - category2
    - category3
    - speaker_type
    - orgtext
    """

    input_path = Path(input_dir).expanduser().resolve()
    output_path = Path(output_file).expanduser().resolve()
    error_path = Path(error_file).expanduser().resolve()

    if not input_path.exists():
        raise FileNotFoundError(f"입력 폴더가 없습니다: {input_path}")

    if not input_path.is_dir():
        raise NotADirectoryError(f"입력 경로가 폴더가 아닙니다: {input_path}")

    json_files = sorted(input_path.rglob("*.json"))

    success_count = 0
    row_count = 0
    error_count = 0
    excluded_file_count = 0
    file_labels = []

    output_path.parent.mkdir(parents=True, exist_ok=True)
    error_path.parent.mkdir(parents=True, exist_ok=True)

    with (
        output_path.open("w", encoding="utf-8-sig", newline="") as out_f,
        error_path.open("w", encoding="utf-8-sig", newline="") as err_f,
    ):
        writer = csv.writer(out_f, delimiter="\t")
        writer.writerow([
            "file_label",
            "category1",
            "category2",
            "category3",
            "speaker_type",
            "orgtext",
        ])

        error_writer = csv.writer(err_f, delimiter="\t")
        error_writer.writerow([
            "json_path",
            "error_type",
            "error_message",
        ])

        for json_path in json_files:
            pending_rows = []

            try:
                with json_path.open("r", encoding="utf-8-sig") as json_f:
                    data = json.load(json_f)

                if not isinstance(data, dict):
                    raise ValueError(
                        f"JSON 최상위 구조가 객체가 아닙니다: "
                        f"{type(data).__name__}"
                    )

                input_texts = data.get("inputText")

                if not isinstance(input_texts, list):
                    raise ValueError(
                        "inputText가 없거나 배열 형식이 아닙니다."
                    )

                if not input_texts:
                    raise ValueError("inputText 배열이 비어 있습니다.")

                info_list = data.get("info")

                if not isinstance(info_list, list) or not info_list:
                    raise ValueError(
                        "info가 없거나 비어 있거나 배열 형식이 아닙니다."
                    )

                first_info = info_list[0]

                if not isinstance(first_info, dict):
                    raise ValueError(
                        "info[0]이 객체 형식이 아닙니다."
                    )

                metadata = first_info.get("metadata")

                if not isinstance(metadata, dict):
                    raise ValueError(
                        "info[0].metadata가 없거나 객체 형식이 아닙니다."
                    )

                category1 = metadata.get("category1")
                category2 = metadata.get("category2")
                category3 = metadata.get("category3")
                speaker_type = metadata.get("speaker_type")

                required_metadata = {
                    "category1": category1,
                    "category2": category2,
                    "category3": category3,
                    "speaker_type": speaker_type,
                }

                for field_name, field_value in required_metadata.items():
                    if field_value is None:
                        raise ValueError(
                            f"metadata.{field_name} 값이 없습니다."
                        )

                    if not isinstance(field_value, str):
                        raise ValueError(
                            f"metadata.{field_name}가 문자열이 아닙니다: "
                            f"{type(field_value).__name__}"
                        )

                    if not field_value.strip():
                        raise ValueError(
                            f"metadata.{field_name} 값이 비어 있습니다."
                        )

                if include_relative_path:
                    file_label = str(json_path.relative_to(input_path))
                else:
                    file_label = json_path.stem

                for item_index, item in enumerate(input_texts):
                    if not isinstance(item, dict):
                        raise ValueError(
                            f"inputText[{item_index}]가 객체 형식이 아닙니다."
                        )

                    orgtext = item.get("orgtext")

                    if orgtext is None:
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 없습니다."
                        )

                    if not isinstance(orgtext, str):
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 "
                            f"문자열이 아닙니다: {type(orgtext).__name__}"
                        )

                    cleaned_text = (
                        orgtext
                        .replace("\t", " ")
                        .replace("\r", " ")
                        .replace("\n", " ")
                        .strip()
                    )

                    if not cleaned_text:
                        raise ValueError(
                            f"inputText[{item_index}].orgtext가 "
                            "빈 문자열입니다."
                        )

                    pending_rows.append([
                        file_label,
                        category1.strip(),
                        category2.strip(),
                        category3.strip(),
                        speaker_type.strip(),
                        cleaned_text,
                    ])

                if not pending_rows:
                    raise ValueError(
                        "저장할 수 있는 orgtext가 없습니다."
                    )

                # 파일 전체 검사가 끝난 뒤에만 최종 TSV에 저장
                writer.writerows(pending_rows)

                file_labels.append(file_label)
                success_count += 1
                row_count += len(pending_rows)

            except (
                json.JSONDecodeError,
                UnicodeDecodeError,
                OSError,
                ValueError,
            ) as e:
                excluded_file_count += 1
                error_count += 1

                print(f"[제외] {json_path}: {e}")

                error_writer.writerow([
                    str(json_path),
                    type(e).__name__,
                    str(e),
                ])

    label_counts = Counter(file_labels)

    duplicate_labels = {
        label: count
        for label, count in label_counts.items()
        if count > 1
    }

    print()
    print(f"입력 폴더: {input_path}")
    print(f"검색한 JSON 파일: {len(json_files):,}개")
    print(f"정상 저장 파일: {success_count:,}개")
    print(f"제외된 파일: {excluded_file_count:,}개")
    print(f"저장 행 수: {row_count:,}개")
    print(f"오류 파일 수: {error_count:,}개")
    print(f"출력 파일: {output_path}")
    print(f"오류 파일: {error_path}")

    if duplicate_labels:
        print(
            f"\n[중복 발견] "
            f"중복 file_label 수: {len(duplicate_labels):,}"
        )

        for label, count in sorted(duplicate_labels.items()):
            print(f"{label}\t{count}")
    else:
        print("\n[정상] 저장된 모든 file_label이 유일합니다.")


### 텍스트 데이터 추출 실행

In [23]:
extract_orgtext_from_folder(
    input_dir=input_dir,
    output_file=output_file,
    error_file=error_file,
    include_relative_path=False,
)

[제외] /home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/02.자살위기개입/03.이성문제/MEN0004835/MEN23000483552A007.json: 'utf-8' codec can't decode byte 0xbe in position 46: invalid start byte
[제외] /home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/02.자살위기개입/03.이성문제/MEN0004835/MEN23000483552A021.json: 'utf-8' codec can't decode byte 0xa4 in position 46: invalid start byte
[제외] /home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/02.자살위기개입/03.이성문제/MEN0004835/MEN23000483552A022.json: 'utf-8' codec can't decode byte 0xb1 in position 46: invalid start byte
[제외] /home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/02.자살위기개입/03.이성문제/MEN0004835/MEN23000483552A026.json: 'utf-8' codec can't decode byte 0xc0 in position 46: invalid start byte
[제외] /home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/02.자살위기개입/03.이성문제/MEN0004835/MEN23000483552A027.json: 'utf-8' codec can't decode byte 0xa4 in position 46: invalid s

In [25]:
import pandas as pd

text_df = pd.read_csv(
    output_file,
    sep="\t",
    encoding="utf-8-sig",
    dtype=str,
)

text_df.columns = (
    text_df.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

for column in [
    "file_label",
    "category1",
    "category2",
    "category3",
    "speaker_type",
    "orgtext",
]:
    text_df[column] = (
        text_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

target_text_df = text_df[
    text_df["category1"].eq("정신건강복지센터")
    & text_df["category2"].eq("자살위기개입")
].copy()

print("=" * 80)
print("전체 전사 행:", len(text_df))
print("전체 file_label:", text_df["file_label"].nunique())
print()
print("선택 카테고리 전사 행:", len(target_text_df))
print("선택 카테고리 file_label:", target_text_df["file_label"].nunique())
print()
print(
    "선택 카테고리 중복 file_label:",
    target_text_df["file_label"].duplicated().sum(),
)
print(
    "빈 전사문:",
    target_text_df["orgtext"].eq("").sum(),
)
print("=" * 80)

print("\ncategory3 실제 분포:")
print(
    target_text_df["category3"]
    .replace("", "[비어 있음]")
    .value_counts(dropna=False)
    .to_string()
)

display(
    target_text_df[
        [
            "file_label",
            "category1",
            "category2",
            "category3",
            "speaker_type",
            "orgtext",
        ]
    ].head(100)
)

전체 전사 행: 2048988
전체 file_label: 2048988

선택 카테고리 전사 행: 236318
선택 카테고리 file_label: 236318

선택 카테고리 중복 file_label: 0
빈 전사문: 0

category3 실제 분포:
category3
가정불화       55643
신체정신적문제    38541
이성문제       36339
외로움고독      36047
경제문제       24309
기타         14259
학교성적진로     11584
친구동료문제     11209
직장문제        8387


,file_label,category1,category2,category3,speaker_type,orgtext
1617198,MEN21000313722A001,정신건강복지센터,자살위기개입,가정불화,상담사,안녕하십니까? ㅇㅇ정신건강복지센터입니다.
1617199,MEN21000313722A002,정신건강복지센터,자살위기개입,가정불화,상담사,궁금하신 점 말씀하십시오.
1617200,MEN21000313722A003,정신건강복지센터,자살위기개입,가정불화,상담사,"네, 잘 하셨어요."
1617201,MEN21000313722A004,정신건강복지센터,자살위기개입,가정불화,상담사,무엇이 답답하실까요?
1617202,MEN21000313722A005,정신건강복지센터,자살위기개입,가정불화,상담사,선생님께서 힘든 일이 있으신 듯 한데.
...,...,...,...,...,...,...
1617293,MEN21000313752B046,정신건강복지센터,자살위기개입,가정불화,고객,이제 고생 끝에 낙이 오나 보다 하고
1617294,MEN21000313752B047,정신건강복지센터,자살위기개입,가정불화,고객,정말 잘 풀렸었어요.
1617295,MEN21000313752B048,정신건강복지센터,자살위기개입,가정불화,고객,"네, 근데 남편이 갑자기 또 일을 벌인 거예요."
1617296,MEN21000313752B049,정신건강복지센터,자살위기개입,가정불화,고객,이번에는 저한테 의논도 없이
